# 02 - Feature Engineering

Build merged train/test datasets using the full aggregation pipeline and save processed artifacts.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import config
from src.aggregation import merge_stage
from src.data_loading import load_application_test, load_application_train, load_auxiliary_tables
from src.feature_engineering import engineer_application_features
from src.preprocessing import replace_day_sentinel
from src.utils import ensure_dir

In [ ]:
df_train = load_application_train()
df_test = load_application_test()
aux = load_auxiliary_tables()

df_train = replace_day_sentinel(df_train, [c for c in df_train.columns if 'DAYS' in c.upper()])
df_test = replace_day_sentinel(df_test, [c for c in df_test.columns if 'DAYS' in c.upper()])

In [ ]:
merged_train = merge_stage(df_train.copy(), 'full', **aux)
merged_train = engineer_application_features(merged_train)

merged_test = merge_stage(df_test.copy(), 'full', **aux)
merged_test = engineer_application_features(merged_test)

print('Merged train shape:', merged_train.shape)
print('Merged test shape:', merged_test.shape)

In [ ]:
ensure_dir(config.PROCESSED_DATA_DIR)
merged_train.to_pickle(config.MERGED_TRAIN_PATH)
merged_test.to_pickle(config.MERGED_TEST_PATH)

print(f'Saved: {config.MERGED_TRAIN_PATH}')
print(f'Saved: {config.MERGED_TEST_PATH}')